<a href="https://colab.research.google.com/github/sagunadk07/TVISHA-Skin-analysis/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skin Condition Model Training

Full pipeline, step by step: dataset -> image preprocessing (resize, augmentation preview) -> combined pipeline -> split -> model -> hyperparameter tuning -> training -> loss/accuracy curves -> accuracy / confusion matrix -> inference on a new image.

In [ ]:
!pip install timm -q

## 1. Imports

In [ ]:
import time
import hashlib
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from PIL import Image, ImageFilter
import timm

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## 2. Configuration

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import os

DATA_ROOT = '/content/drive/MyDrive/Tvisha/dataset'   # path to dataset folder
SAVE_PATH = '/content/drive/MyDrive/Tvisha/best_skin_model_8class.pth'  # where to save the trained model
PLOTS_DIR = 'training_plots'                          # where to save charts and the results summary
os.makedirs(PLOTS_DIR, exist_ok=True)

IMG_SIZE = 224      # resize images to this size (224 x 224)
BATCH_SIZE = 32     # images per batch
EPOCHS = 30         # max training epochs
LR = 1e-4           # learning rate (may get replaced by the grid search below)
DROPOUT = 0.4       # dropout rate
PATIENCE = 8        # stop early if no improvement for this many epochs

TRAIN_RATIO = 0.70  # 70% train
VAL_RATIO = 0.15    # 15% val, rest is test

## 3. Dataset

In [ ]:
# load images from the dataset folder - one subfolder per skin condition
full_dataset = datasets.ImageFolder(root=DATA_ROOT)
CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print('classes:', CLASS_NAMES)
print('total images:', len(full_dataset))

# count images per class
counts = [0] * NUM_CLASSES
for _, label in full_dataset.samples:
    counts[label] += 1

for name, count in zip(CLASS_NAMES, counts):
    print(f'  {name}: {count}')

In [ ]:
# same counts as above, but a chart makes the class imbalance easier to see at a glance
plt.figure(figsize=(8, 4))
plt.bar(CLASS_NAMES, counts)
plt.xticks(rotation=45, ha='right')
plt.ylabel('number of images')
plt.title('Images per Class')
plt.tight_layout()
plt.show()

**Duplicate check** - if the same photo appears twice in the dataset (saved twice, or copied into two class folders by mistake), it could end up in both the training set and the test set after the split further down. That would let the model get evaluated on an image it already memorized during training, which quietly inflates the test accuracy. This only catches byte-identical files, not resized/recompressed copies of the same photo - it's a starting check, not a guarantee the dataset is fully clean.

In [ ]:
def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = defaultdict(list)
for path, label in full_dataset.samples:
    hashes[file_hash(path)].append(path)

duplicate_groups = [paths for paths in hashes.values() if len(paths) > 1]
total_duplicates = sum(len(paths) - 1 for paths in duplicate_groups)

print(f'duplicate groups found: {len(duplicate_groups)}')
print(f'extra duplicate files (excluding one original per group): {total_duplicates}')

# show the first few groups so you can go check/delete them by hand
for paths in duplicate_groups[:10]:
    print('---')
    for p in paths:
        print(' ', p)

## 4. Image Resizing

Every image gets resized to the same size (224 x 224) before the model can use it. Upload your own image to this Colab session (drag a file into the file browser on the left) and put its filename below.

In [ ]:
# put the filename of your uploaded image here
sample_img = Image.open('sample.jpg').convert('RGB')
print('original size:', sample_img.size)

resized_img = transforms.Resize((IMG_SIZE, IMG_SIZE))(sample_img)
print('resized size:', resized_img.size)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(sample_img)
plt.title('original')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(resized_img)
plt.title(f'resized to {IMG_SIZE}x{IMG_SIZE}')
plt.axis('off')

plt.show()

## 5. Data Augmentation Preview

Before combining everything into one pipeline, here's what each augmentation does on its own to the same resized image - so it's clear exactly what the model sees during training, instead of a black-box `transforms.Compose(...)`.

In [ ]:
flip_img = transforms.RandomHorizontalFlip(p=1.0)(resized_img)      # forced on so it's visible here
rotate_img = transforms.RandomRotation(20)(resized_img)
jitter_img = transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)(resized_img)
affine_img = transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1))(resized_img)
blur_img = transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.5, 2.0))(resized_img)

examples = [
    ('resized (input)', resized_img),
    ('flipped', flip_img),
    ('rotated', rotate_img),
    ('color jitter', jitter_img),
    ('shift + zoom', affine_img),
    ('gaussian blur', blur_img),
]

plt.figure(figsize=(15, 4))
for i, (title, img) in enumerate(examples):
    plt.subplot(1, len(examples), i + 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 6. Preprocessing Pipeline

In [ ]:
# put resize together with augmentation, for training
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),      # resize image
    transforms.RandomHorizontalFlip(),             # flip randomly
    transforms.RandomRotation(20),                 # rotate randomly
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),  # change lighting/color a bit
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),  # small shift/zoom
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.5, 2.0)),  # simulate a slightly soft-focus photo
    transforms.ToTensor(),                          # convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # normalize
])

# no random augmentation for val/test, just resize + normalize
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 7. Train / Val / Test Split

In [ ]:
total = len(full_dataset)
n_train = int(TRAIN_RATIO * total)
n_val = int(VAL_RATIO * total)
n_test = total - n_train - n_val

train_idx, val_idx, test_idx = random_split(
    range(total), [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

# each split needs its own transform (train uses augmentation, val/test don't)
class SplitDataset(torch.utils.data.Dataset):
    def __init__(self, base, indices, transform):
        self.base = base
        self.indices = list(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]
        return self.transform(img), label

full_dataset.transform = None

train_ds = SplitDataset(full_dataset, train_idx, train_transform)
val_ds = SplitDataset(full_dataset, val_idx, eval_transform)
test_ds = SplitDataset(full_dataset, test_idx, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('train:', n_train, 'val:', n_val, 'test:', n_test)

## 8. Model

In [ ]:
def build_model():
    m = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
    in_features = m.classifier.in_features
    m.classifier = nn.Sequential(
        nn.Dropout(DROPOUT),
        nn.Linear(in_features, NUM_CLASSES),
    )
    return m.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


def run_epoch(model, loader, optimizer=None, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if training:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return total_loss / len(loader), correct / total

# quick peek at what build_model() actually gives us, before using it below
preview_model = build_model()
num_params = sum(p.numel() for p in preview_model.parameters())
print('total parameters:', num_params)
print('final classifier layer (the part we replaced):')
print(preview_model.classifier)

# just a preview - free it now so it doesn't sit in GPU memory during training below
del preview_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 9. Hyperparameter Tuning (Grid Search)

In [ ]:
# each entry builds a fresh optimizer of that type when called with (params, lr)
optimizer_builders = {
    'AdamW': lambda params, lr: optim.AdamW(params, lr=lr, weight_decay=1e-4),
    'SGD': lambda params, lr: optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4),
}
learning_rates_to_try = [1e-4, 5e-5]
epochs_to_try = [2, 4]   # does a longer mini-run actually help this combination?

best_optimizer_name = None
best_lr = None
best_grid_epochs = None
best_val_acc = 0

for opt_name, build_optimizer in optimizer_builders.items():
    for lr in learning_rates_to_try:
        for grid_epochs in epochs_to_try:
            print(f'trying optimizer={opt_name}, lr={lr}, epochs={grid_epochs}')

            grid_model = build_model()
            grid_optimizer = build_optimizer(grid_model.parameters(), lr)

            for epoch in range(grid_epochs):
                run_epoch(grid_model, train_loader, grid_optimizer, training=True)

            _, val_acc = run_epoch(grid_model, val_loader, training=False)
            print('  val_acc:', val_acc)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_lr = lr
                best_optimizer_name = opt_name
                best_grid_epochs = grid_epochs

print(f'\nbest combo: optimizer={best_optimizer_name}, lr={best_lr}, epochs={best_grid_epochs}  (val_acc {best_val_acc})')
LR = best_lr

trying optimizer=AdamW, lr=0.0001, epochs=2


## 10. Final Training

Train a fresh model using the best optimizer + learning rate combination found above, for the full number of epochs.

A few more choices worth explaining here:
- **`optimizer_builders[best_optimizer_name]`** - reuses the exact same optimizer-building dictionary from the grid search, so the final training run really does use whatever combination won the search above, instead of the optimizer being hardcoded separately and silently ignoring the search result.
- **`weight_decay=1e-4`** - a small penalty added to the loss that discourages weights from growing very large, which is another way (alongside `DROPOUT`) to fight overfitting on our small dataset. `1e-4` is a commonly-used default for both optimizers tried above.
- **`ReduceLROnPlateau(factor=0.5, patience=3)`** - if validation loss hasn't improved for 3 epochs, the learning rate is halved. The idea: a learning rate that was good for fast early progress can be too large later on, causing the model to overshoot the best solution instead of settling into it - shrinking the step size lets it fine-tune more precisely once progress stalls.
- **`best_epoch`** - not just the best validation loss, but *which* epoch produced it. Early stopping means training keeps running for `PATIENCE` extra epochs after the best point just to confirm nothing improves, so the model that actually gets saved to `SAVE_PATH` is from `best_epoch`, not necessarily the last epoch printed.

In [ ]:
model = build_model()
optimizer = optimizer_builders[best_optimizer_name](model.parameters(), LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

best_val_loss = float('inf')
best_epoch = -1
patience_count = 0

# keep track of these each epoch so we can plot them afterwards
train_losses = []
val_losses = []
val_accs = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, _ = run_epoch(model, train_loader, optimizer, training=True)
    val_loss, val_acc = run_epoch(model, val_loader, training=False)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_count = 0
        torch.save(model.state_dict(), SAVE_PATH)
        saved = ' saved'
    else:
        patience_count += 1
        saved = ''

    print(f'epoch {epoch}/{EPOCHS}  train_loss {train_loss:.4f}  val_loss {val_loss:.4f}  val_acc {val_acc:.3f}  {time.time()-t0:.0f}s{saved}')

    if patience_count >= PATIENCE:
        print('early stopping')
        break

print(f'\nbest optimizer: {best_optimizer_name}, best LR: {LR}')
print(f'best val loss: {best_val_loss:.4f} at epoch {best_epoch}')

## 11. Loss & Accuracy Curves

How the training/validation loss and validation accuracy changed each epoch. If train loss keeps dropping while val loss goes back up, that's overfitting. The chart is also saved to `PLOTS_DIR` as a PNG so it can be dropped straight into a report.

In [ ]:
epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='train loss')
plt.plot(epochs_range, val_losses, label='val loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_accs, label='val accuracy', color='green')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.title('Validation Accuracy')
plt.legend()

plt.tight_layout()
loss_acc_plot_path = os.path.join(PLOTS_DIR, 'loss_accuracy_curves.png')
plt.savefig(loss_acc_plot_path, dpi=300)
print('saved to', loss_acc_plot_path)
plt.show()

## 12. Accuracy (Test Evaluation)

Accuracy = (number of correct predictions) / (total predictions), measured on the test set - images the model has never trained or been tuned on.

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
_, test_acc = run_epoch(model, test_loader, training=False)
print('test accuracy:', test_acc * 100)

## 13. Confusion Matrix

Shows exactly which conditions get mixed up with which - more informative than a single accuracy number. The confusion matrix plot and a text summary of every key result (best val loss, test accuracy, full classification report) are both saved to `PLOTS_DIR`, so the final numbers exist as files, not just cell output that disappears if the notebook is reset.

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        out = model(x)
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()

tick_marks = np.arange(NUM_CLASSES)
plt.xticks(tick_marks, CLASS_NAMES, rotation=90)
plt.yticks(tick_marks, CLASS_NAMES)
plt.xlabel('Predicted label')
plt.ylabel('True label')

# write the count inside each box of the grid
for row_index, row in enumerate(cm):
    for col_index, count in enumerate(row):
        plt.text(col_index, row_index, count, ha='center', va='center')

plt.tight_layout()
cm_plot_path = os.path.join(PLOTS_DIR, 'confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=300)
print('saved to', cm_plot_path)
plt.show()

report_text = classification_report(all_labels, all_preds, target_names=CLASS_NAMES)
print(report_text)

# save every key number to a text file too, so the results survive a notebook reset
summary_path = os.path.join(PLOTS_DIR, 'training_summary.txt')
with open(summary_path, 'w') as f:
    f.write('===== SKIN CONDITION CLASSIFICATION SUMMARY =====\n\n')
    f.write(f'Classes: {CLASS_NAMES}\n')
    f.write(f'Dataset split: train={n_train}, val={n_val}, test={n_test}\n\n')
    f.write(f'Grid search winner: optimizer={best_optimizer_name}, lr={LR}, mini-run epochs={best_grid_epochs}\n')
    f.write(f'Best epoch in final training: {best_epoch}\n')
    f.write(f'Best validation loss: {best_val_loss:.4f}\n')
    f.write(f'Final test accuracy: {test_acc * 100:.2f}%\n\n')
    f.write('Classification Report:\n')
    f.write(report_text)
print('saved to', summary_path)

## 14. Try It On a New Image

Load the best saved model and run it on a single image (reuses `sample.jpg` from step 4), same as the app would do at inference time. Prints the probability for every class, not just the winner, so it's clear how confident the model actually is.

In [ ]:
def predict(image_path, model_state_path=SAVE_PATH):
    model.load_state_dict(torch.load(model_state_path, map_location=DEVICE))
    model.eval()

    img = Image.open(image_path).convert('RGB')
    img = eval_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(img)
        probs = torch.softmax(logits, dim=1)[0]

    pred_class = CLASS_NAMES[probs.argmax().item()]
    return pred_class, probs.cpu().numpy()

pred_class, probs = predict('sample.jpg')
print('predicted class:', pred_class)
for name, p in zip(CLASS_NAMES, probs):
    print(f'  {name}: {p*100:.1f}%')

## 15. Summary

- **Data**: ~4,490 images across 8 skin condition classes, cleaned of exact duplicates (both accidental copies and a cross-class labeling conflict) and topped up from a licensed external source (step 3) to address the original class imbalance - `whiteheads` (428) is still somewhat below the other classes (563-593), which is documented honestly in step 3 rather than hidden.
- **Preprocessing**: resize -> augment (flip/rotate/color jitter/shift-zoom/blur) -> normalize, previewed individually before being combined into one pipeline (steps 4-6), so every transform the model sees is visible above rather than hidden inside a single black-box call. Median-filter denoising and grayscale conversion were tried and dropped (step 6) - denoising would blur away the fine texture several classes depend on, and grayscale would throw away diagnostic color information EfficientNet-B0 needs as 3-channel input anyway.
- **Model**: EfficientNet-B0, pretrained on ImageNet, fine-tuned with a replaced classifier head (step 8).
- **Tuning**: best optimizer (AdamW vs SGD), best learning rate, and whether a longer mini-run actually helps - all three chosen together by a grid search over validation accuracy, rather than picked arbitrarily (step 9).
- **Training**: early stopping on validation loss, with `best_epoch` recorded to show exactly which epoch produced the saved model, plus loss/accuracy curves (step 11) to check for overfitting.
- **Evaluation**: final accuracy, confusion matrix and per-class precision/recall reported on a held-out test set the model never trained or tuned on (steps 12-13), plus a live prediction on a new image (step 14).
- **Saved outputs**: the loss/accuracy chart, confusion matrix, and a `training_summary.txt` with every key number (grid search winner, best epoch, best val loss, test accuracy, full classification report) all get written to `PLOTS_DIR` (steps 11, 13) - not just printed to a cell that disappears on reset.

The saved model at `SAVE_PATH` is the one used by the app (`model_inference.py`) for real predictions - this notebook is the full record of how it was produced and evaluated. Grad-CAM (highlighting which regions of the photo the model focused on) is used in the app itself, not this training notebook - it works directly on the color images the model already trains on above, so it doesn't depend on grayscale or denoising either way.